# Encoder baseline on track 1: DeBERTa-v3-base, three seeds

The deep-learning baseline for Assignment 1: the organizers' pilot model family, fine-tuned by our own seeded loop (`dico_nli.baselines.encoder`) on Modal GPUs, then scored here with the official scorer.

**What this notebook does**

1. Reads the three seed runs fetched from the Modal volume (`predictions.csv`, `probabilities.csv`, `run.json` per seed).
2. Scores each seed's dev predictions with the official scorer and reports mean and standard deviation over seeds.
3. Draws the training curves, one confusion matrix, the comparison of every Assignment 1 baseline, and the accuracy-against-consistency view.
4. Records the run: seeds, hyperparameters, GPU, timing, data commit, command.

**Training protocol, fixed before the first launch**

| Setting | Value |
|---|---|
| Model | `microsoft/deberta-v3-base`, sequence-pair classification head |
| Input | `(premise, hypothesis)` tokenized as one pair, max 32 tokens |
| Optimisation | AdamW, learning rate 2e-5, weight decay 0.01, batch 16, 5 epochs, linear schedule with 6 % warm-up, gradient clipping at 1.0 |
| Epoch selection | best weighted F1 on a 20 % slice of train, split by source pair with the run's seed; dev is never seen during fitting |
| Seeds | 13, 42, 1234; each seed also draws its own train/holdout split |
| Compute | one T4 per seed on Modal, launched by `scripts/modal/train_encoder.py` |

**Where the results go**

| Output | Path |
|---|---|
| Dev scores per seed and summary | `experiments/2026-09-29-deberta-base-track1/tables/` |
| Predictions and scorer reports per seed | `experiments/2026-09-29-deberta-base-track1/predictions/`, `scores/seed<seed>/` |
| Per-seed training records from Modal | `experiments/2026-09-29-deberta-base-track1/runs/seed<seed>.json` |
| Figures | `experiments/2026-09-29-deberta-base-track1/figures/*.pdf`, with `.png` previews |
| Run record and results table | `experiments/2026-09-29-deberta-base-track1/config.json`, `results.md` |
| Written findings | `experiments/2026-09-29-deberta-base-track1/findings.md` |

**How to run**

The GPU part runs on Modal (see the launcher's docstring for the launch and fetch commands). With the fetched runs under `artifacts/dl-baseline-track1/<RUN_ID>/`, this notebook runs on any machine:

```bash
PYTHONIOENCODING=utf-8 uv run jupyter execute notebooks/assignment-1/04_dl_baseline_track1.ipynb
```

## 1. Setup

Imports and the run parameters. `RUN_ID` is the folder name used on the Modal volume and under `artifacts/`.

In [ ]:
import json
import shutil
from datetime import date

import matplotlib
matplotlib.use("Agg")
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Image, display

from dico_nli.data import DATA_ROOT, LABELS, REPO_ROOT, load_reference, read_instances
from dico_nli.experiments import experiment_dir, write_config, write_results_table
from dico_nli.plotting import INK, INK_SECONDARY, LABEL_COLORS, SEQUENTIAL, figure, paper_theme, save_figure
from dico_nli.scoring import Scores, score, write_predictions

TRACK = 1
RUN_ID = "2026-09-29-deberta-base-track1"
SEEDS = [13, 42, 1234]
RUN_DATE = date(2026, 9, 29)
OVERWRITE = True
FETCHED = REPO_ROOT / "artifacts" / "dl-baseline-track1" / RUN_ID
SHORT = {"EQUIVALENCE": "Equiv.", "FORWARD_ENTAILMENT": "Forward",
         "BACKWARD_ENTAILMENT": "Backward", "NEGATIVE_OTHER": "Neg./other"}
ORDER = list(LABELS)
SERIES = ["#2a78d6", "#eb6834", "#1baf7a"]  # one validated categorical slot per seed
paper_theme()

## 2. Data and fetched runs

Dev and its reference file for the scorer, the run folder, and a check that every seed's files are present.

In [ ]:
dev = load_reference(TRACK, "dev")
gold = DATA_ROOT / "dev" / f"dico_nli_dev_track{TRACK}_reference.csv"
out = experiment_dir("deberta-base-track1", day=RUN_DATE, overwrite=OVERWRITE)
tables, figures, runs = out / "tables", out / "figures", out / "runs"
for folder in (tables, runs):
    folder.mkdir(exist_ok=True)

records = {}
for seed in SEEDS:
    folder = FETCHED / f"seed{seed}"
    missing = [name for name in ("predictions.csv", "probabilities.csv", "run.json") if not (folder / name).exists()]
    assert not missing, f"seed {seed}: missing {missing} under {folder}"
    records[seed] = json.loads((folder / "run.json").read_text(encoding="utf-8"))
    shutil.copy(folder / "run.json", runs / f"seed{seed}.json")


def show(fig, stem):
    pdf_path, png_path = save_figure(fig, figures / stem)
    display(Image(filename=str(png_path)))


print({seed: (r["gpu"], r["train_seconds"], r["best_epoch"]) for seed, r in records.items()})

## 3. Score every seed on dev

Each seed's predictions are copied into the run folder in the official format and scored by the official scorer. The summary reports the mean and standard deviation over the three seeds.

In [ ]:
rows, per_seed = {}, []
for seed in SEEDS:
    predictions = pd.read_csv(FETCHED / f"seed{seed}" / "predictions.csv")
    mapping = dict(zip(predictions["instance_id"], predictions["label"]))
    prediction_path = write_predictions(out / "predictions" / f"seed{seed}.csv", mapping)
    result = score(gold, prediction_path, out / "scores" / f"seed{seed}")
    rows[f"deberta-v3-base, seed {seed}"] = result
    report = json.loads(result.report_path.read_text(encoding="utf-8"))
    per_seed.append({
        "seed": seed, "best_epoch": records[seed]["best_epoch"],
        "holdout_weighted_f1": records[seed]["history"][records[seed]["best_epoch"] - 1]["holdout_weighted_f1"],
        "weighted_f1": result.weighted_f1, "soft_cons": result.soft_cons, "hard_cons": result.hard_cons,
        "macro_f1": report["classification"]["macro_f1"], "accuracy": report["classification"]["accuracy"],
        "reversible_pairs": report["consistency"]["reversible_pairs"],
        "soft_consistent_pairs": report["consistency"]["soft_consistent_pairs"],
        "hard_correct_pairs": report["consistency"]["hard_correct_pairs"],
        "train_seconds": records[seed]["train_seconds"], "gpu": records[seed]["gpu"],
    })
per_seed = pd.DataFrame(per_seed)
per_seed.to_csv(tables / "dev_scores_per_seed.csv", index=False)
summary = per_seed[["weighted_f1", "soft_cons", "hard_cons", "macro_f1", "accuracy"]].agg(["mean", "std"]).round(4)
summary.to_csv(tables / "dev_scores_summary.csv")
rows["deberta-v3-base, mean of 3 seeds"] = Scores(summary.loc["mean", "weighted_f1"], summary.loc["mean", "soft_cons"],
                                                   summary.loc["mean", "hard_cons"], tables / "dev_scores_summary.csv")
display(per_seed.round(4))
summary

## 4. Per-label scores

Precision, recall and F1 per label, averaged over the seeds, from the scorer's own reports.

In [ ]:
per_label = []
for seed in SEEDS:
    report = json.loads((out / "scores" / f"seed{seed}" / "scores.json").read_text(encoding="utf-8"))
    for label, stats in report["classification"]["per_label"].items():
        per_label.append({"seed": seed, "label": label, "precision": stats["precision"], "recall": stats["recall"], "f1": stats["f1"]})
per_label = pd.DataFrame(per_label)
per_label_summary = per_label.groupby("label")[["precision", "recall", "f1"]].agg(["mean", "std"]).reindex(ORDER).round(3)
per_label_summary.to_csv(tables / "dev_per_label.csv")
per_label_summary

## 5. Figures

**Figure 1.** Held-out weighted F1 per epoch for each seed; the marker is the epoch that was kept.

In [ ]:
fig, ax = figure(columns=1, height=2.1)
for colour, seed in zip(SERIES, SEEDS):
    history = pd.DataFrame(records[seed]["history"])
    ax.plot(history["epoch"], history["holdout_weighted_f1"], color=colour, linewidth=1.2, marker="o", markersize=3, label=f"seed {seed}")
    best = records[seed]["best_epoch"]
    ax.plot([best], [history.loc[history["epoch"] == best, "holdout_weighted_f1"].iloc[0]], color=colour, marker="o",
            markersize=7, markeredgecolor="white", markeredgewidth=1.2)
ax.set_xlabel("epoch")
ax.set_ylabel("held-out weighted F1")
ax.set_xticks(sorted({e for r in records.values() for e in [h["epoch"] for h in r["history"]]}))
ax.set_title("Epoch selection on the 20 % train slice")
ax.legend(loc="lower right", handlelength=1.2)
show(fig, "f1_training_curves")

**Figure 2.** Confusion matrix on dev for the first seed. Rows are gold labels, columns are predictions.

In [ ]:
seed = SEEDS[0]
predictions = pd.read_csv(out / "predictions" / f"seed{seed}.csv")
gold_labels = {i.instance_id: i.label for i in dev}
confusion = pd.crosstab(pd.Series([gold_labels[k] for k in predictions["instance_id"]], name="gold"),
                        pd.Series(list(predictions["label"]), name="predicted")).reindex(index=ORDER, columns=ORDER).fillna(0).astype(int)
confusion.to_csv(tables / f"dev_confusion_seed{seed}.csv")

fig, ax = figure(columns=1, height=2.6)
sns.heatmap(confusion, ax=ax, cmap=sns.light_palette(SEQUENTIAL[5], as_cmap=True), annot=True, fmt="d", cbar=False,
            linewidths=1.5, linecolor="white", annot_kws={"size": 7},
            xticklabels=[SHORT[l] for l in ORDER], yticklabels=[SHORT[l] for l in ORDER])
ax.set_xlabel("predicted")
ax.set_ylabel("gold")
ax.set_title(f"DeBERTa-v3-base, seed {seed}, on dev")
ax.grid(False)
ax.tick_params(length=0)
show(fig, "f2_confusion")

**Figure 3.** Every Assignment 1 baseline on track 1 dev. The trivial and ML numbers are read from their own recorded runs; the encoder bar is the mean over seeds with the standard deviation as an error bar.

In [ ]:
def official(path):
    report = json.loads(path.read_text(encoding="utf-8"))
    return {"weighted_f1": report["weighted_f1"], "soft_cons": report["soft_cons"], "hard_cons": report["hard_cons"]}

systems = {
    "majority": official(REPO_ROOT / "experiments" / "2026-09-29-trivial-baselines-track1" / "scores" / "majority" / "scores.json"),
    "random": official(REPO_ROOT / "experiments" / "2026-09-29-trivial-baselines-track1" / "scores" / "random" / "scores.json"),
    "cues + logreg": official(REPO_ROOT / "experiments" / "2026-09-29-ml-baseline-track1" / "scores" / "cues+logreg" / "scores.json"),
    "both + logreg": official(REPO_ROOT / "experiments" / "2026-09-29-ml-baseline-track1" / "scores" / "both+logreg" / "scores.json"),
    "DeBERTa-v3-base": {m: summary.loc["mean", m] for m in ("weighted_f1", "soft_cons", "hard_cons")},
}
errors = {m: summary.loc["std", m] for m in ("weighted_f1", "soft_cons", "hard_cons")}
comparison = pd.DataFrame(systems).T
comparison.to_csv(tables / "all_baselines_dev.csv")

metrics = [("weighted_f1", "Weighted F1", SEQUENTIAL[1]), ("soft_cons", "SoftCons", SEQUENTIAL[3]), ("hard_cons", "HardCons", SEQUENTIAL[5])]
fig, ax = figure(columns=1, height=2.8)
y = np.arange(len(comparison))
for k, (column, name, colour) in enumerate(metrics):
    values = comparison[column].to_numpy()
    err = [errors[column] if s == "DeBERTa-v3-base" else 0 for s in comparison.index]
    ax.barh(y + (k - 1) * 0.26, values, height=0.24, color=colour, label=name, xerr=err, error_kw={"elinewidth": 0.8, "ecolor": INK})
    for yy, v in zip(y + (k - 1) * 0.26, values):
        ax.text(v + 0.03, yy, f"{v:.2f}", va="center", fontsize=6.5, color=INK_SECONDARY)
ax.set_yticks(y)
ax.set_yticklabels(comparison.index)
ax.invert_yaxis()
ax.set_xlim(0, 1.15)
ax.set_xticks([0, 0.25, 0.5, 0.75, 1.0])
ax.grid(False, axis="y")
ax.grid(True, axis="x")
ax.set_title("Assignment 1 baselines on track 1 dev")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=3, handlelength=1.0)
show(fig, "f3_all_baselines")

**Figure 4.** Accuracy against consistency: weighted F1 on the x axis, SoftCons on the y axis, one point per system (the encoder's three seeds and their mean). A system that gains F1 without losing consistency moves right without moving down.

In [ ]:
points = comparison[["weighted_f1", "soft_cons"]].copy()
for seed, row in per_seed.set_index("seed").iterrows():
    points.loc[f"seed {seed}"] = [row["weighted_f1"], row["soft_cons"]]

fig, ax = figure(columns=1, height=2.6)
for name, (x, yv) in points.iterrows():
    is_seed = name.startswith("seed")
    ax.scatter([x], [yv], s=14 if is_seed else 26, color=SEQUENTIAL[2] if is_seed else SEQUENTIAL[5], edgecolor="white", linewidth=0.8, zorder=3)
    if not is_seed:
        ax.annotate(name, (x, yv), xytext=(4, 4), textcoords="offset points", fontsize=6.5, color=INK_SECONDARY)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xlabel("weighted F1")
ax.set_ylabel("SoftCons")
ax.grid(True, axis="both")
ax.set_title("Accuracy against self-consistency, track 1 dev")
show(fig, "f4_f1_vs_softcons")

## 6. Run record

`config.json` records the seeds, the training settings, the GPU and timing per seed, the data commit, this repository's commit and the command. `results.md` holds the per-seed and mean scores in the form the report cites.

In [ ]:
write_config(
    out, seed=SEEDS[0], track=TRACK,
    command="PYTHONIOENCODING=utf-8 uv run jupyter execute notebooks/assignment-1/04_dl_baseline_track1.ipynb",
    extra={"seeds": SEEDS, "run_id": RUN_ID, "training": records[SEEDS[0]]["config"],
           "holdout_fraction": records[SEEDS[0]]["holdout_fraction"],
           "gpu": {str(s): r["gpu"] for s, r in records.items()},
           "train_seconds": {str(s): r["train_seconds"] for s, r in records.items()},
           "best_epoch": {str(s): r["best_epoch"] for s, r in records.items()},
           "launcher": "scripts/modal/train_encoder.py"},
)
print(write_results_table(out, rows, title=f"DeBERTa-v3-base, track {TRACK} dev").read_text(encoding="utf-8"))